# Entrenamiento de YOLO26 sobre el dataset proxy (iteración 1)

Un solo cuaderno para todas las corridas de `docs/experimentos.md`, parte 1.5. Lo único que cambia entre corridas es la **primera celda**. El resto sigue el protocolo de la sección 1.3 y no se edita.

| Corrida | `RUN_ID` | `MODEL` | `DEGREES` | `SEED` |
|---|---|---|---|---|
| E1 | `e1_n_deg0_s0` | `yolo26n.pt` | 0 | 0 |
| E2 | `e2_n_deg10_s0` | `yolo26n.pt` | 10 | 0 |
| E1b | `e1b_n_deg0_s1` | `yolo26n.pt` | 0 | 1 |
| E3 | `e3_s_degX_s0` | `yolo26s.pt` | según ablación | 0 |
| E4 | `e4_m_degX_s0` | `yolo26m.pt` | según ablación | 0 |

Configuración del cuaderno: GPU **T4 x2** (se usa una, `device=0`), **Internet activado**, dataset `deteccionoso` adjunto. Para corridas largas, lanzarlo con *Save Version > Save & Run All (Commit)*: corre en segundo plano y las salidas quedan guardadas aunque se cierre el navegador.

## 0. Parámetros de la corrida (única celda que se edita)

In [ ]:
RUN_ID  = "e1_n_deg0_s0"   # identificador de la corrida, ver tabla de arriba
MODEL   = "yolo26n.pt"     # yolo26n.pt | yolo26s.pt | yolo26m.pt
DEGREES = 0                # 0 o 10 (ablación de rotación)
SEED    = 0                # 0, salvo E1b que usa 1
EPOCHS  = 100              # protocolo: 100. Solo bajar para pruebas

print(f"Corrida {RUN_ID}: {MODEL}, degrees={DEGREES}, seed={SEED}, epochs={EPOCHS}")

## 1. Entorno

In [ ]:
%pip install -q ultralytics==8.4.138

import sys, torch, ultralytics
print("Python       :", sys.version.split()[0])
print("torch        :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0), "| n GPUs:", torch.cuda.device_count())
print("Ultralytics  :", ultralytics.__version__)
try:
    import albumentations
    print("albumentations:", albumentations.__version__, "(Ultralytics añadirá Blur, MedianBlur, ToGray y CLAHE con p=0.01)")
except ImportError:
    print("albumentations: no instalado")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Dataset

Esperado: train 2656/2656, val 598/598, test 578/578. Listas: val_inat 260, val_ena24 338, test_inat 260, test_ena24 318.

In [ ]:
from pathlib import Path

DATASET = Path("/kaggle/input/datasets/josemartinvallejos/deteccionoso")
assert DATASET.is_dir(), f"No existe {DATASET}. Revisar que el dataset esté adjunto."
for conj in ("train", "val", "test"):
    n_img = sum(1 for _ in (DATASET / "images" / conj).glob("*.jpg"))
    n_lbl = sum(1 for _ in (DATASET / "labels" / conj).glob("*.txt"))
    print(f"{conj:<6} {n_img} imágenes, {n_lbl} etiquetas")
for lista in ("val_inat.txt", "val_ena24.txt", "test_inat.txt", "test_ena24.txt"):
    n = sum(1 for _ in open(DATASET / lista, encoding="utf-8"))
    print(f"{lista:<16} {n} rutas")

## 3. Archivos yaml con la ruta de Kaggle

In [ ]:
CFG = Path("/kaggle/working/cfg")
CFG.mkdir(parents=True, exist_ok=True)

def escribir_yaml(nombre, val, test):
    ruta = CFG / nombre
    ruta.write_text(
        f"path: {DATASET}\ntrain: images/train\nval: {val}\ntest: {test}\nnc: 1\nnames: ['oso']\n",
        encoding="utf-8",
    )
    return str(ruta)

YAML = {
    "global":     escribir_yaml("dataset.yaml",            "images/val",    "images/test"),
    "val_inat":   escribir_yaml("dataset_val_inat.yaml",   "val_inat.txt",  "val_inat.txt"),
    "val_ena24":  escribir_yaml("dataset_val_ena24.yaml",  "val_ena24.txt", "val_ena24.txt"),
    "test_inat":  escribir_yaml("dataset_test_inat.yaml",  "test_inat.txt", "test_inat.txt"),
    "test_ena24": escribir_yaml("dataset_test_ena24.yaml", "test_ena24.txt","test_ena24.txt"),
}
print(open(YAML["global"]).read())

## 4. Entrenamiento

Protocolo 1.3 más `degrees` y `seed` de la celda 0. Qué mirar en la salida: las líneas `Scanning` (2656 y 598, 0 corrupt), la línea `optimizer:` (debe ser AdamW lr=0.002) y la línea `albumentations:`.

In [ ]:
import time
from ultralytics import YOLO

RUNS = "/kaggle/working/runs"
t0 = time.time()
model = YOLO(MODEL)
model.train(
    data=YAML["global"],
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    seed=SEED,
    deterministic=True,
    degrees=DEGREES,
    cache="ram",
    device=0,
    project=RUNS,
    name=RUN_ID,
    exist_ok=True,
)
TIEMPO_MIN = (time.time() - t0) / 60
print(f"\nTiempo total de la celda: {TIEMPO_MIN:.1f} min")

## 5. Validación en tres particiones con `best.pt`

Solo validación. El conjunto de prueba no se toca hasta cerrar el modelo.

In [ ]:
import pandas as pd

RUN_DIR = Path(RUNS) / RUN_ID
BEST = RUN_DIR / "weights" / "best.pt"
m = YOLO(str(BEST))
filas = []
for nombre in ("global", "val_ena24", "val_inat"):
    r = m.val(data=YAML[nombre], imgsz=640, batch=16, device=0, plots=False,
              project=RUNS, name=f"{RUN_ID}_val_{nombre}", exist_ok=True)
    d = r.results_dict
    filas.append({
        "corrida": RUN_ID,
        "partición": nombre,
        "P": d["metrics/precision(B)"],
        "R": d["metrics/recall(B)"],
        "mAP50": d["metrics/mAP50(B)"],
        "mAP50-95": d["metrics/mAP50-95(B)"],
    })
tabla = pd.DataFrame(filas).round(4)
print(tabla.to_string(index=False))

## 6. Resumen y archivos para descargar

Deja en `/kaggle/working/` cuatro archivos con el prefijo de la corrida: `best.pt`, `results.csv`, `args.yaml` y la tabla de validación, más un zip con las gráficas de Ultralytics (sin pesos). Copiar la salida impresa en la entrada correspondiente de `docs/experimentos.md`.

In [ ]:
import shutil

OUT = Path("/kaggle/working")
shutil.copy(BEST, OUT / f"{RUN_ID}_best.pt")
shutil.copy(RUN_DIR / "results.csv", OUT / f"{RUN_ID}_results.csv")
shutil.copy(RUN_DIR / "args.yaml", OUT / f"{RUN_ID}_args.yaml")
tabla.to_csv(OUT / f"{RUN_ID}_val_por_particion.csv", index=False)

# gráficas y salidas de la corrida, sin la carpeta de pesos
tmp = Path("/tmp") / f"{RUN_ID}_run"
if tmp.exists():
    shutil.rmtree(tmp)
shutil.copytree(RUN_DIR, tmp, ignore=shutil.ignore_patterns("weights"))
shutil.make_archive(str(OUT / f"{RUN_ID}_run"), "zip", tmp)

res = pd.read_csv(RUN_DIR / "results.csv")
res.columns = [c.strip() for c in res.columns]
mejor = res["metrics/mAP50-95(B)"].idxmax()
print(f"Corrida        : {RUN_ID}  ({MODEL}, degrees={DEGREES}, seed={SEED}, epochs={EPOCHS})")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"Épocas hechas  : {len(res)}")
print(f"Tiempo         : {res['time'].iloc[-1] / 60:.1f} min de entrenamiento, "
      f"{res['time'].iloc[-1] / len(res):.1f} s/época, {TIEMPO_MIN:.1f} min la celda completa")
print(f"Mejor época    : {int(res.loc[mejor, 'epoch'])} "
      f"(mAP50-95 val = {res.loc[mejor, 'metrics/mAP50-95(B)']:.4f}, "
      f"mAP50 = {res.loc[mejor, 'metrics/mAP50(B)']:.4f})")
print(f"Última época   : mAP50-95 val = {res['metrics/mAP50-95(B)'].iloc[-1]:.4f}")
print()
print(tabla.to_string(index=False))
print()
print("Archivos para descargar:")
for p in sorted(OUT.glob(f"{RUN_ID}_*")):
    print(f"   {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")